# Notebook 09: Quantization

## Why This Matters
LLaMA-70B in full FP32 precision requires ~280 GB of GPU memory — more than three A100 80GB cards just for weights. In INT4, it fits in one. Quantization is the primary tool for making large models accessible on practical hardware, and understanding it is essential for anyone deploying, benchmarking, or selecting models for production.

## Structure
1. Background: what quantization is and why the community adopted it
2. How quantization works: the math of precision reduction
3. INT8 quantization — the first practical tier
4. INT4 quantization — the sweet spot for consumer hardware
5. GGUF and llama.cpp quantization formats
6. Quality vs compression tradeoff curves
7. KV cache quantization
8. Practical guidance: which format for which use case

## What You'll Learn
- The math of how float16 values get mapped to int8/int4
- Why quantization error is not uniform — and which layers are sensitive
- The difference between weight-only and weight+activation quantization
- How to read GGUF quantization names (Q4_K_M, Q8_0, etc.)
- Memory savings and quality tradeoffs at each tier

## Background

### What is quantization?

Neural network weights are typically stored as 32-bit or 16-bit floating-point numbers. Quantization reduces this precision — storing weights as 8-bit integers, 4-bit integers, or even fewer bits. Less precision per weight means smaller memory footprint, faster memory reads, and often faster arithmetic on hardware that has efficient low-precision compute units.

The core tradeoff: fewer bits = less memory and faster inference, but more rounding error = worse model quality. The art of quantization is finding compression levels where quality loss is acceptable for your use case.

### Why the community needed quantization

For the first few years of the transformer era (2017–2022), quantization was a niche concern — model sizes were manageable and hardware was relatively abundant. GPT-3 (175B parameters) in 2020 changed the conversation. At FP16, GPT-3 requires ~350 GB of GPU memory, putting it far beyond what a single machine could serve.

The real inflection point was **LLaMA** (Meta, 2023). Meta released weights for 7B, 13B, 33B, and 65B parameter models — open weights that researchers and developers could actually run. But even the 7B model required ~14 GB in FP16, meaning a single consumer GPU (most of which top out at 8–16 GB VRAM) couldn't load it. The pressure to run capable models on accessible hardware drove a burst of quantization research.

### The bitsandbytes and GGUF moments

Two tools transformed the landscape in 2023:

**bitsandbytes** (Tim Dettmers et al.) introduced practical INT8 inference for transformers. The key paper, LLM.int8() (2022), showed that most of a model's weight matrix is low-magnitude and quantizes well, but a small fraction of "emergent outlier" features have much larger magnitudes and cause large quantization errors. The solution: identify these outlier dimensions and keep them in FP16 while quantizing the rest to INT8. This *mixed-precision* approach preserved quality while halving memory.

**GGUF** (Georgi Gerganov, llama.cpp) took a different approach: aggressive INT4 and even INT3/INT2 quantization with per-block scaling factors, targeting CPU inference on consumer hardware. A Q4_K_M quantized LLaMA-7B fits in ~4 GB — runnable on a MacBook Pro with 8 GB RAM. This democratized local LLM inference in a way that hadn't been possible before.

**GPTQ** (Frantar et al., 2022) added a third approach: post-training quantization that uses a calibration dataset to minimize quantization error layer by layer. Better quality than naive rounding, especially at aggressive compression ratios.

### The quality story

The practical finding across dozens of papers and benchmarks: **INT8 is nearly lossless** for most tasks. Quality drop from FP16 → INT8 is typically well under 1% on standard benchmarks. **INT4** introduces noticeable but often acceptable degradation — roughly equivalent to dropping to a model 1–2 sizes smaller. **INT3 and below** starts to degrade significantly.

The right framing: quantization lets you run a larger model at lower precision rather than a smaller model at full precision. A Q4 LLaMA-70B often outperforms a full-precision LLaMA-13B despite requiring similar memory, because the capacity advantage of 70B parameters outweighs the precision loss.

In [ ]:
%pip install -q torch numpy

In [ ]:
import math
import torch
import numpy as np
import torch.nn.functional as F

torch.manual_seed(42)
print(f"PyTorch: {torch.__version__}")

## 1. How Quantization Works: The Math

Quantization maps a range of floating-point values to a fixed set of integers. The simplest form is **absmax quantization** (symmetric):

```
scale = max(|W|) / (2^(n_bits - 1) - 1)   ← how many float units per integer step
W_q = round(W / scale)                      ← quantize
W_deq = W_q × scale                         ← dequantize (approximate reconstruction)

For INT8: integers range from -127 to 127 (127 steps per direction)
For INT4: integers range from -7 to 7   (7 steps per direction)
```

**Quantization error** = `W_deq - W` — the information lost by rounding. The key insight: this error is bounded by `scale / 2` (half a step). Smaller scale (more compressed value range) → smaller error. Outlier values force a large scale → larger error for all other values.

In [ ]:
def absmax_quantize(w: torch.Tensor, n_bits: int) -> tuple:
    """Symmetric absmax quantization. Returns (quantized, scale, dequantized)."""
    max_val = 2 ** (n_bits - 1) - 1  # e.g., 127 for int8, 7 for int4
    scale = w.abs().max() / max_val
    w_q = (w / scale).round().clamp(-max_val, max_val).to(torch.int8)
    w_deq = w_q.float() * scale
    return w_q, scale, w_deq


def zero_point_quantize(w: torch.Tensor, n_bits: int) -> tuple:
    """Asymmetric zero-point quantization — handles non-symmetric distributions."""
    n_levels = 2 ** n_bits
    w_min, w_max = w.min(), w.max()
    scale = (w_max - w_min) / (n_levels - 1)
    zero_point = (-w_min / scale).round().clamp(0, n_levels - 1)
    w_q = ((w / scale) + zero_point).round().clamp(0, n_levels - 1)
    w_deq = (w_q - zero_point) * scale
    return w_q, scale, zero_point, w_deq


# Compare on a small weight tensor
w = torch.tensor([-1.5, -0.8, -0.2, 0.0, 0.3, 0.7, 1.1, 1.8])

print("Original weights:", w.numpy().round(3))
print()

for n_bits in [8, 4, 3, 2]:
    w_q, scale, w_deq = absmax_quantize(w, n_bits)
    error = (w - w_deq).abs()
    print(f"INT{n_bits} absmax:")
    print(f"  scale={scale:.4f}, range=[{-2**(n_bits-1)+1}, {2**(n_bits-1)-1}]")
    print(f"  quantized: {w_q.numpy()}")
    print(f"  dequantized: {w_deq.numpy().round(3)}")
    print(f"  max error: {error.max():.4f}, mean error: {error.mean():.4f}")
    print()

## 2. The Outlier Problem

The critical practical challenge in quantizing large transformers is **outlier features** — a small fraction of weight dimensions with dramatically larger magnitudes than the rest.

These outliers exist in almost all large transformers (>6.7B parameters). They appear at consistent positions across layers — the same few dimensions are always large. The problem: absmax quantization must accommodate the maximum value, which forces all other values to use only a tiny fraction of the integer range, greatly increasing quantization error for the majority of weights.

```
Weight distribution example:
  99.9% of values: range [-1.5, 1.5]
  0.1% of values:  range [-500, 500]   ← outliers

Absmax scale for INT8: 500 / 127 = 3.94
Step size: 3.94  ← one INT8 step = 3.94 float units

For a value of 0.8 (typical):
  Quantized: round(0.8 / 3.94) = round(0.20) = 0
  Dequantized: 0 × 3.94 = 0.0
  Error: 0.8  ← catastrophic!

Without outliers, scale ≈ 0.012 → step size 0.012 → error ≈ 0.006 for the same value.
```

This is why naive INT8 quantization of large models fails, and why solutions like LLM.int8() and GPTQ specifically handle outliers.

In [ ]:
def simulate_outlier_impact(n_weights: int = 1000, outlier_fraction: float = 0.001,
                             outlier_magnitude: float = 100.0, n_bits: int = 8):
    """Demonstrate how outliers degrade quantization quality for normal weights."""
    torch.manual_seed(0)
    # Normal weights: mostly small values
    w_normal = torch.randn(n_weights) * 0.5

    # Add outliers
    n_outliers = max(1, int(n_weights * outlier_fraction))
    outlier_idx = torch.randperm(n_weights)[:n_outliers]
    w_with_outliers = w_normal.clone()
    w_with_outliers[outlier_idx] = outlier_magnitude * torch.randn(n_outliers)

    # Quantize both
    _, _, w_no_outlier_deq = absmax_quantize(w_normal, n_bits)
    _, scale_with, w_with_outlier_deq = absmax_quantize(w_with_outliers, n_bits)

    # Measure error on NORMAL weights only (outliers are expected to be imprecise)
    normal_mask = torch.ones(n_weights, dtype=torch.bool)
    normal_mask[outlier_idx] = False

    err_clean = (w_normal - w_no_outlier_deq).abs()
    err_contaminated = (w_with_outliers[normal_mask] - w_with_outlier_deq[normal_mask]).abs()

    print(f"Impact of {n_outliers} outlier(s) (magnitude={outlier_magnitude}) on INT{n_bits} quantization:")
    print(f"  Without outliers:  scale={w_normal.abs().max()/(2**(n_bits-1)-1):.4f}")
    print(f"  With outliers:     scale={scale_with:.4f}  ({scale_with / (w_normal.abs().max()/(2**(n_bits-1)-1)):.0f}× larger)")
    print(f"  Normal weight error without outliers: mean={err_clean.mean():.5f}")
    print(f"  Normal weight error WITH outliers:    mean={err_contaminated.mean():.5f}  "
          f"({err_contaminated.mean()/err_clean.mean():.0f}× worse)")

simulate_outlier_impact()
print()
simulate_outlier_impact(outlier_magnitude=500.0)

## 3. Per-Channel and Per-Block Quantization

The outlier problem motivated finer-grained quantization scales. Instead of one scale for the entire matrix:

**Per-channel (per-row) quantization**: compute a separate scale per output channel (row) of the weight matrix. Outliers in one channel don't contaminate the scale for other channels. Used in LLM.int8().

**Per-block quantization**: divide the weight matrix into small blocks (e.g., 32 or 128 values), compute a scale per block. Used in GGUF and GPTQ. Better than per-channel for very low bit widths (INT4, INT3).

```
Absmax (one scale for whole matrix):
  [scale = max(|all weights|)]
  [-1.2, 0.3, ..., 487.2, ..., -0.5]  ← 487.2 poisons the scale for everyone

Per-block (scale per 32-value group):
  Block 0: [scale = 1.4] → [-1.2, 0.3, ..., 1.1]  ← tight scale, low error
  Block k: [scale = 487] → [..., 487.2, ...]        ← outlier only affects its block
```

In [ ]:
def per_block_quantize(w: torch.Tensor, n_bits: int, block_size: int = 32) -> tuple:
    """Per-block absmax quantization."""
    flat = w.flatten()
    n = len(flat)
    n_blocks = math.ceil(n / block_size)

    scales = []
    w_q_all = []
    w_deq_all = []

    for b in range(n_blocks):
        start, end = b * block_size, min((b + 1) * block_size, n)
        block = flat[start:end]
        w_q, scale, w_deq = absmax_quantize(block, n_bits)
        scales.append(scale)
        w_q_all.append(w_q)
        w_deq_all.append(w_deq)

    return torch.cat(w_q_all).reshape(w.shape), scales, torch.cat(w_deq_all).reshape(w.shape)


# Compare global vs per-block quantization on a tensor with outliers
torch.manual_seed(0)
w_test = torch.randn(256)
w_test[42] = 150.0  # inject an outlier at position 42

_, _, global_deq = absmax_quantize(w_test, n_bits=8)
_, block_scales, block_deq = per_block_quantize(w_test, n_bits=8, block_size=32)

# Exclude the outlier position from error measurement
mask = torch.ones(256, dtype=torch.bool)
mask[42] = False

global_err = (w_test[mask] - global_deq[mask]).abs()
block_err  = (w_test[mask] - block_deq[mask]).abs()

print("INT8 quantization: global scale vs per-block (block_size=32)")
print(f"  Global scale:          {w_test.abs().max()/(127):.4f}")
print(f"  Per-block scales:      {[f'{s:.4f}' for s in block_scales[:4]]} ... (varies per block)")
print()
print(f"  Global error (excl. outlier):    mean={global_err.mean():.5f}, max={global_err.max():.5f}")
print(f"  Per-block error (excl. outlier): mean={block_err.mean():.5f},  max={block_err.max():.5f}")
print(f"  Improvement: {global_err.mean()/block_err.mean():.1f}× better mean error")

## 4. Memory Savings: The Practical Numbers

The memory equation for a model's weights:

```
Memory = n_parameters × bytes_per_parameter

FP32:  4 bytes/param
BF16:  2 bytes/param  (standard for training and inference)
INT8:  1 byte/param   (2× reduction vs BF16)
INT4:  0.5 bytes/param (4× reduction vs BF16)
INT3:  0.375 bytes/param
INT2:  0.25 bytes/param
```

In practice, INT4/INT3 quantization stores additional metadata (scales, zero points) per block, adding a small overhead (~3% for block_size=32 at INT4).

In [ ]:
def model_memory_gb(n_params: int, bits: int, block_size: int = 32) -> float:
    """Model memory in GB including quantization metadata."""
    weight_bits = n_params * bits
    # Per-block scales stored in FP16 (16 bits each)
    n_blocks = n_params / block_size
    scale_bits = n_blocks * 16
    total_bits = weight_bits + scale_bits
    return total_bits / 8 / 1e9


models = [
    ("LLaMA-7B",   7e9),
    ("LLaMA-13B",  13e9),
    ("LLaMA-33B",  33e9),
    ("LLaMA-70B",  70e9),
    ("LLaMA-405B", 405e9),
]

precisions = [("FP32", 32), ("BF16", 16), ("INT8", 8), ("INT4", 4), ("INT3", 3)]

print("Model weight memory (GB):")
header = f"{'Model':<14}" + "".join(f"{name:>9}" for name, _ in precisions)
print(header)
print("-" * (14 + 9 * len(precisions)))

for model_name, n_params in models:
    row = f"{model_name:<14}"
    for prec_name, bits in precisions:
        mem = model_memory_gb(n_params, bits)
        row += f"{mem:>8.1f}G"
    print(row)

print()
print("Consumer GPU VRAM for reference:")
for gpu, vram in [("RTX 3060", 12), ("RTX 4090", 24), ("M3 Max 128GB", 128), ("A100 80GB", 80)]:
    print(f"  {gpu}: {vram} GB")
print()
print("Key insight: INT4 LLaMA-70B (~37 GB) fits on two A100s or a Mac Studio.")
print("FP16 LLaMA-70B (~140 GB) requires four A100s.")

## 5. GGUF Quantization Names Decoded

GGUF (the format used by llama.cpp, Ollama, and most local inference tools) uses a naming convention that's worth understanding:

```
Q4_K_M
│ │ │ └── Size variant: S=small, M=medium, L=large
│ │ └──── K = uses k-quants (mixed precision, quality-optimized)
│ └────── Number of bits for most weights
└──────── Q = quantized

Q8_0    → 8-bit, method 0 (simple absmax per block)
Q6_K    → 6-bit k-quants
Q5_K_M  → 5-bit k-quants, medium size
Q4_K_M  → 4-bit k-quants, medium size  ← most popular, good quality/size balance
Q4_K_S  → 4-bit k-quants, small size
Q3_K_M  → 3-bit k-quants, medium
Q2_K    → 2-bit k-quants
```

**K-quants** (introduced by Georgi Gerganov in llama.cpp) use mixed precision: some weight matrices (especially attention output projections and FFN down projections, which are more sensitive) are kept at higher precision, while less sensitive matrices use lower precision. This improves quality significantly at the same average bit width compared to uniform quantization.

In [ ]:
# GGUF format sizes for LLaMA-7B
# (approximate file sizes from Hugging Face model cards)
gguf_sizes = {
    "F16 (baseline)": {"gb": 13.0, "ppl_delta": 0.00, "bits": 16.0},
    "Q8_0":           {"gb": 6.67, "ppl_delta": 0.03, "bits": 8.0},
    "Q6_K":           {"gb": 5.15, "ppl_delta": 0.08, "bits": 6.0},
    "Q5_K_M":         {"gb": 4.45, "ppl_delta": 0.15, "bits": 5.0},
    "Q4_K_M":         {"gb": 3.80, "ppl_delta": 0.28, "bits": 4.0},
    "Q4_K_S":         {"gb": 3.56, "ppl_delta": 0.45, "bits": 4.0},
    "Q3_K_M":         {"gb": 2.95, "ppl_delta": 0.82, "bits": 3.0},
    "Q2_K":           {"gb": 2.33, "ppl_delta": 2.10, "bits": 2.0},
}

print("LLaMA-7B GGUF quantization comparison:")
print(f"{'Format':<16} {'Size':>8} {'vs F16':>8} {'PPL delta':>11} {'Quality verdict'}")
print("-" * 65)
f16_size = gguf_sizes["F16 (baseline)"]["gb"]
for name, info in gguf_sizes.items():
    size_ratio = info["gb"] / f16_size
    ppl = info["ppl_delta"]
    if ppl < 0.05:
        verdict = "Lossless in practice"
    elif ppl < 0.3:
        verdict = "Near-lossless"
    elif ppl < 0.8:
        verdict = "Mild degradation"
    elif ppl < 1.5:
        verdict = "Noticeable degradation"
    else:
        verdict = "Significant degradation"
    print(f"{name:<16} {info['gb']:>7.2f}G {size_ratio:>7.1%} {ppl:>+10.2f}  {verdict}")

print()
print("PPL delta = perplexity increase vs F16 (lower is better).")
print("Rule of thumb: Q4_K_M is the best quality/size tradeoff for most use cases.")
print("Q8_0 if you have the VRAM and want near-perfect quality.")

## 6. Quality vs Compression Tradeoff Curves

The practical question is always: **how much quality do I sacrifice for how much memory savings?**

The tradeoff is non-linear:
- FP32 → BF16: essentially zero quality loss, 2× memory reduction (almost always worth it)
- BF16 → INT8: ~0.03 perplexity increase, 2× memory reduction (almost always worth it)
- INT8 → INT4: ~0.2–0.5 perplexity increase, 2× memory reduction (usually worth it — bigger model wins)
- INT4 → INT3: quality drops more steeply (diminishing returns on compression)
- INT3 → INT2: substantial degradation (only for extreme memory constraints)

In [ ]:
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for saving
import matplotlib.pyplot as plt

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

# Quality-compression frontier (LLaMA-7B approximation from published benchmarks)
formats = list(gguf_sizes.keys())
sizes_gb = [gguf_sizes[f]["gb"] for f in formats]
ppl_deltas = [gguf_sizes[f]["ppl_delta"] for f in formats]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("LLaMA-7B: Quality vs Compression Tradeoff", fontsize=12)

# Plot 1: size vs quality
ax1.plot(sizes_gb, ppl_deltas, 'o-', color='steelblue', linewidth=2)
for name, s, p in zip(formats, sizes_gb, ppl_deltas):
    ax1.annotate(name.split()[0], (s, p), textcoords="offset points", xytext=(5, 3), fontsize=8)
ax1.set_xlabel("Model size (GB)")
ax1.set_ylabel("Perplexity increase vs FP16")
ax1.set_title("Size vs Quality")
ax1.axhline(y=0.3, color='orange', linestyle='--', alpha=0.7, label="~0.3 PPL threshold")
ax1.legend(fontsize=8)
ax1.grid(alpha=0.3)

# Plot 2: compression ratio vs quality loss rate
compression_ratios = [f16_size / s for s in sizes_gb]
ax2.bar(range(len(formats)), ppl_deltas, color='steelblue', alpha=0.8)
ax2.set_xticks(range(len(formats)))
ax2.set_xticklabels([f.split()[0] for f in formats], rotation=45, ha='right')
ax2.set_ylabel("Perplexity delta vs FP16")
ax2.set_title("Quality Loss by Format")
ax2.axhline(y=0.3, color='orange', linestyle='--', alpha=0.7)
ax2.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig("quantization_tradeoffs.png", bbox_inches='tight')
plt.show()
print("Saved: quantization_tradeoffs.png")

## 7. Bigger Model at Lower Precision vs Smaller Model at Full Precision

The key practical insight from quantization research: **model scale usually beats precision**.

When choosing between:
- LLaMA-13B at FP16 (~26 GB)
- LLaMA-70B at Q4_K_M (~37 GB)

The 70B Q4 model nearly always wins on benchmarks, despite being quantized. The capacity advantage of 70B parameters outweighs the precision loss from INT4. The 13B model simply doesn't have enough parameters to represent certain patterns, regardless of precision.

In [ ]:
# Model selection guide given a VRAM budget
# Approximate sizes: model_name → {bits: size_gb}
model_configs = [
    ("LLaMA-7B",   {16: 14.0, 8: 7.0,  4: 3.8}),
    ("LLaMA-13B",  {16: 26.0, 8: 13.0, 4: 7.0}),
    ("LLaMA-33B",  {16: 66.0, 8: 33.0, 4: 18.0}),
    ("LLaMA-70B",  {16: 140.0, 8: 70.0, 4: 37.0}),
    ("Mixtral-8x7B", {16: 90.0, 8: 45.0, 4: 24.0}),  # MoE model
]

vram_budgets = [8, 16, 24, 48, 80]

print("Best model for VRAM budget (favoring larger model over higher precision):")
print(f"{'VRAM Budget':>12} | {'Best option'}")
print("-" * 55)

for vram in vram_budgets:
    options = []
    for model_name, sizes in model_configs:
        for bits, size in sorted(sizes.items(), reverse=True):  # prefer higher precision
            if size <= vram:
                options.append((model_name, bits, size))
                break  # take the highest precision that fits

    if options:
        # Pick largest model that fits (model scale > precision)
        best = max(options, key=lambda x: int(x[0].split('-')[1].replace('B','').replace('x7','56')))
        print(f"{vram:>11}GB | {best[0]} at {'BF16' if best[1]==16 else f'INT{best[1]}'} ({best[2]:.0f}GB)")
    else:
        print(f"{vram:>11}GB | No model fits")

print()
print("General rule: run the largest model that fits, at the lowest precision that still fits.")

## 8. KV Cache Quantization

Weight quantization reduces the model's static memory. But for long-context serving, the *KV cache* can equal or exceed the model weight memory (see notebook 05). Quantizing the KV cache is a separate optimization.

- **KV cache in FP16**: baseline, full precision
- **KV cache in INT8**: 2× reduction, minimal quality impact (KV values are activations, not weights — they have different sensitivity profiles)
- **KV cache in INT4**: 4× reduction, slightly more quality impact — still often acceptable

vLLM supports INT8 KV cache quantization natively. This can be the difference between fitting 8 vs 16 concurrent requests in memory.

In [ ]:
def total_serving_memory_gb(
    model_params: int,
    weight_bits: int,
    n_layers: int,
    d_model: int,
    batch_size: int,
    seq_len: int,
    kv_bits: int = 16,
) -> dict:
    weights = model_params * weight_bits / 8 / 1e9
    kv_cache = 2 * n_layers * batch_size * seq_len * d_model * kv_bits / 8 / 1e9
    return {"weights_gb": weights, "kv_cache_gb": kv_cache, "total_gb": weights + kv_cache}


print("Total serving memory: LLaMA-7B, batch=16, seq=2048")
print(f"{'Weight quant':>14} {'KV quant':>10} {'Weights':>10} {'KV cache':>10} {'Total':>10} {'Max concurrent (80GB GPU)':>26}")
print("-" * 80)

for w_bits, kv_bits in [(16, 16), (8, 16), (8, 8), (4, 16), (4, 8), (4, 4)]:
    m = total_serving_memory_gb(7e9, w_bits, 32, 4096, 16, 2048, kv_bits)
    # How many concurrent requests at 2048 tokens fit in 80GB?
    weights = 7e9 * w_bits / 8 / 1e9
    kv_per_request = 2 * 32 * 1 * 2048 * 4096 * kv_bits / 8 / 1e9
    max_concurrent = int((80 - weights) / kv_per_request)
    print(f"{f'INT{w_bits}':>14} {f'INT{kv_bits}':>10} {m['weights_gb']:>9.1f}G "
          f"{m['kv_cache_gb']:>9.1f}G {m['total_gb']:>9.1f}G {max_concurrent:>26}")

## Summary

**What quantization is:** Reducing weight precision from FP16/FP32 to INT8/INT4 to trade accuracy for memory efficiency.

**Why the community adopted it:** Models grew faster than GPU memory. INT4 LLaMA-70B fits on hardware that can't hold FP16 LLaMA-13B — and performs better.

**The quality story:**
```
FP32 → BF16:  Lossless in practice              (always do this)
BF16 → INT8:  Near-lossless (+0.03 PPL)         (almost always worth it)
INT8 → INT4:  Mild degradation (+0.2-0.5 PPL)   (worth it if it unlocks a larger model)
INT4 → INT3:  Noticeable degradation            (only if necessary)
INT3 → INT2:  Significant degradation           (last resort)
```

**The practical rule:** Larger model at lower precision almost always beats smaller model at higher precision. Run the biggest model that fits.

**GGUF shorthand:** Q4_K_M = 4-bit k-quants, medium size = the community's go-to for local inference.

**Next:** Notebook 10 covers speculative decoding — using a small draft model to propose tokens that a large target model verifies, achieving 2–3× decode speedup with zero quality loss.